- **HW1 (Czech 2021 Census — vzdělání × pohlaví × věk, 4×2×2).** Load `data/csu_education.csv` (16 rows, ~8.3 M observations). Variables: `vzdelani` ∈ {ZS, SSnoM, SSM, VS}, `pohlavi` ∈ {muz, zena}, `vek` ∈ {15-64, 65+}.
  1. Crosstab the data and compute marginal proportions of each education level by sex, then conditional on each age group. **Notice the cohort reversal**: among 15–64 women hold more tertiary degrees than men, but in 65+ the pattern flips. This is a real-world Simpson-style hint we will revisit in ex12 §3.6.
  2. Fit each of: mutual independence, all three partial-independence variants, and all three conditional-independence variants. Tabulate deviance, df, AIC. Stop short of uniform association and saturated — those wait for ex12.
  3. Discuss which two-way associations are strongest. Tie back to the cohort reversal.
  4. Find the simplest acceptable model from the seven fitted; justify by LRT and AIC.

  Source: ČSÚ, *Výsledky sčítání 2021 — otevřená data*. Aggregation script in `_builders/_make_csu_education.py`.


In [1]:
import pandas as pd
import requests
import os
import io
import statsmodels.formula.api as smf
import statsmodels.api as sm
from scipy.stats import chi2

In [3]:
import sys

# helpers.py: load it from local file when running outside Colab,
# otherwise fetch the GitHub raw version.
try:
    from helpers import Anova
except ImportError:
    if "google.colab" in sys.modules:
        !pip install wget
        import wget
        wget.download(
            "https://github.com/francji1/01ZLMA/raw/main/code/helpers.py",
            "helpers.py",
        )
    else:
        sys.path.insert(0, ".")
    from helpers import Anova

In [4]:
BASE_URL = "https://raw.githubusercontent.com/francji1/01ZLMA/main/data/"


def load_csv(name: str, sep: str = ",") -> pd.DataFrame:
    """Load a CSV from the project data folder.

    Tries the GitHub raw URL first (Colab path); on a 404 falls back to
    a sibling ``data/`` directory next to the notebook (local path,
    useful when the dataset has been added but not yet pushed).
    """
    r = requests.get(BASE_URL + name, verify=False, timeout=30)
    if r.status_code == 200:
        return pd.read_csv(io.StringIO(r.text), sep=sep)
    local = os.path.join(os.path.dirname(os.path.abspath("__file__")), "..", "data", name)
    if os.path.exists(local):
        return pd.read_csv(local, sep=sep)
    r.raise_for_status()  # surface the original HTTP error

In [5]:
edu = load_csv("csu_education.csv")
edu

c:\Users\francji1\AppData\Local\Programs\Python\Python312\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'raw.githubusercontent.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


,vzdelani,pohlavi,vek,pocet
0,ZS,muz,15-64,411015
1,ZS,muz,65+,60769
2,ZS,zena,15-64,411312
3,ZS,zena,65+,280864
4,SSnoM,muz,15-64,1125130
5,SSnoM,muz,65+,429106
6,SSnoM,zena,15-64,759257
7,SSnoM,zena,65+,423490
8,SSM,muz,15-64,1000401
9,SSM,muz,65+,223476


## 1. Crosstab the data and marginal proportions

In [6]:
edu.pivot_table(index=["vek","pohlavi"], columns="vzdelani", values="pocet",
                 aggfunc="sum", margins=True)

vzdelani           SSM    SSnoM       VS       ZS      All
vek   pohlavi                                             
15-64 muz      1000401  1125130   624409   411015  3160955
      zena     1141485   759257   802402   411312  3114456
65+   muz       223476   429106   146045    60769   859396
      zena      376327   423490   105541   280864  1186222
All            2741689  2736983  1678397  1163960  8321029

In [7]:
marginal = edu.groupby(['pohlavi', 'vzdelani'])['pocet'].sum().unstack()
marginal_pct = marginal.div(marginal.sum(axis=1), axis=0) * 100

print("Marginální proporce (Celkově)")
print(marginal_pct.round(2))
print("\n")

conditional = edu.groupby(['vek', 'pohlavi', 'vzdelani'])['pocet'].sum().unstack()

conditional_pct = conditional.div(conditional.sum(axis=1), axis=0) * 100

print("Podmíněné proporce (rozdělené podle věku)")
print(conditional_pct.round(2))

Marginální proporce (Celkově)
vzdelani    SSM  SSnoM     VS     ZS
pohlavi                             
muz       30.44  38.66  19.16  11.73
zena      35.29  27.50  21.11  16.09


Podmíněné proporce (rozdělené podle věku)
vzdelani         SSM  SSnoM     VS     ZS
vek   pohlavi                            
15-64 muz      31.65  35.59  19.75  13.00
      zena     36.65  24.38  25.76  13.21
65+   muz      26.00  49.93  16.99   7.07
      zena     31.72  35.70   8.90  23.68


## 2. Partial vs. Conditional Independence

In [8]:
# A...Age(vek)
# E...Education(vzdelani)
# S...Sex(pohlavi)

### a) mutual independence

In [9]:
# 0. Vzdělání, pohlaví a věk jsou vzájemně nezávislé
mutind  = smf.glm("pocet ~ vzdelani + pohlavi + vek", data=edu,
                     family=sm.families.Poisson()).fit()

### b) partial independence

In [10]:
# 1. Pohlaví a věk jsou závislé, ale vzdělání je nezávislé
partind_SA_E = smf.glm("pocet ~ pohlavi * vek + vzdelani", data=edu,
                       family=sm.families.Poisson()).fit()

# 2. Vzdělání a pohlaví jsou závislé, ale věk je ni nich nezávislý
partind_ES_A = smf.glm("pocet ~ vzdelani * pohlavi + vek", data=edu,
                       family=sm.families.Poisson()).fit()

# 3. Vzdělání a věk jsou závislé, ale pohlaví je na nich nezávislé
partind_EA_S = smf.glm("pocet ~ vzdelani * vek + pohlavi", data=edu,
                       family=sm.families.Poisson()).fit()

### c) conditional independence

In [11]:
# 4. Vzdělání a věk jsou nezávislé pro dané pohlaví
condind_age_edu = smf.glm("pocet ~ vzdelani*pohlavi + pohlavi*vek", data=edu,
                     family=sm.families.Poisson()).fit()

# 5. Vzdělání a pohlaví jsou nezávislé pro danou věkovou kategorii
condind_edu_sex = smf.glm("pocet ~ vzdelani*vek + vek*pohlavi", data=edu,
                     family=sm.families.Poisson()).fit()

# 6. Věk a pohlaví jsou nezávislé pro danou úroveň dosaženého vzdělání
condind_age_sex = smf.glm("pocet ~ vzdelani*pohlavi + vzdelani*vek", data=edu,
                     family=sm.families.Poisson()).fit()

### Shrnutí všech modelů

In [12]:
models = [
    ("mutind A⟂E⟂S", mutind),
    ("partind S⟂E & A⟂E", partind_SA_E),
    ("partind E⟂A & S⟂A", partind_ES_A),
    ("partind E⟂S & A⟂S", partind_EA_S),
    ("condind E⟂A|S", condind_age_edu),
    ("condind E⟂S|A", condind_edu_sex),
    ("condind S⟂A|E", condind_age_sex)
]

tab_edu = pd.DataFrame({
    "model":    [m[0] for m in models],
    "df":       [m[1].df_resid for m in models],
    "deviance": [m[1].deviance for m in models],
    "AIC":      [m[1].aic for m in models]
})

display(tab_edu.round().astype(int, errors='ignore'))

,model,df,deviance,AIC
0,mutind A⟂E⟂S,10,471283,471531
1,partind S⟂E & A⟂E,9,427943,428193
2,partind E⟂A & S⟂A,7,345319,345572
3,partind E⟂S & A⟂S,7,293305,293558
4,condind E⟂A|S,6,301979,302235
5,condind E⟂S|A,6,249965,250220
6,condind S⟂A|E,4,167340,167600


## 3. Strongest associations

Na základě hodnot deviací a AIC v předchozí tabulce se zdá, že nejsilnější vazba je mezi vzděláním a věkem.

Po přidání interakce Vzdělání x Věk, klesla deviance oproti modelu bez interakcí (tedy za předpokladu, že jsou všechny prediktory nezávislé) nejvíce oproti ostatním možným interakcím. Druhý největší pokles deviance nastal po přidání interakce Vzdělání x Pohlaví.

Po kombinaci těchto dvou nejsilnějších interakcí do modelu podmíněné závislosti jsme dospěli k největšímu snížení deviance (z původních 471 283 na 167 340).

## 4. Simplest acceptable model

Vyjdeme z modelu condind_age_sex $(S\perp A|E)$, který se zdá být podle deviace a AIC nejlepší a porovnáme ho pomocí LRT testu s vnořenými modely partind_ES_A $(E\perp A \wedge S\perp A)$ a partind_EA_S $(E\perp S \wedge A \perp S)$.

In [13]:
anova = Anova()

In [14]:
display(anova(partind_ES_A, condind_age_sex, test="LRT"))

LRT — Likelihood Ratio Test  (Lecture 5: deviační test, statistika T_1)
    T_1 = (D_0 - D) / phi  ~  chi^2(p - p_0)        [phi: 1.0  (Poisson / Binomial: phi known and fixed)]
    aliases: LRT = chi-squared deviance test = T_1


,resid_df,resid_deviance,df,deviance,LRT,p_val
0,7,345318.892869,NaN,NaN,NaN,NaN
1,4,167340.371592,3.0,177978.521277,177978.521277,0.0


In [15]:
display(anova(partind_EA_S, condind_age_sex, test="LRT"))

LRT — Likelihood Ratio Test  (Lecture 5: deviační test, statistika T_1)
    T_1 = (D_0 - D) / phi  ~  chi^2(p - p_0)        [phi: 1.0  (Poisson / Binomial: phi known and fixed)]
    aliases: LRT = chi-squared deviance test = T_1


,resid_df,resid_deviance,df,deviance,LRT,p_val
0,7,293304.649511,NaN,NaN,NaN,NaN
1,4,167340.371592,3.0,125964.277918,125964.277918,0.0


U obou testů nám vyšla p-hodnota prakticky nulová, což svědčí o nezbytnosti zahrnutí interakce pohlaví se vzděláním a rovněž věku s pohlavím do modelu.

### Závěr
Ze sedmi nafitovaných modelů se přikláníme k použití modelu condind_age_sex $(S\perp A|E)$, který vykazuje nejnižší hodnoty deviace a AIC. Možnost vynechání některé interakce byla zamítnuta na základě provedeného LRT-testu. Větší počet parametrů našeho modelu by však, s ohledem na množství dat, které máme k dispozici, neměl být problém.